# Chapter 1 — Mathematical Foundations

**Math for ML Engineers** · companion notebook

This notebook mirrors §1.4 (SVD) and §1.5 (gradients of quadratic forms) of the book, the two pieces of linear algebra you will reach for constantly once you start reading or writing training code — low-rank factorizations show up anywhere a weight matrix gets compressed or adapted, and quadratic-form gradients are the algebra underneath every least-squares and regularization term you will differentiate by hand or check numerically.

**Key takeaway:** the SVD is not just a decomposition to memorize — it is the tool that tells you the best possible low-rank approximation of a matrix, and it is the same tool (in spirit) that makes LoRA fine-tuning of large language models tractable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

np.random.seed(0)
np.set_printoptions(precision=4, suppress=True)

## SVD and the Eckart–Young theorem

Every matrix $A \in \mathbb{R}^{m \times n}$ admits a singular value decomposition

$$
A = U \Sigma V^\top = \sum_{i=1}^{r} \sigma_i \, u_i v_i^\top ,
$$

with $U$ and $V$ orthogonal, $\Sigma$ diagonal with non-negative entries $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r \ge 0$, and $r = \mathrm{rank}(A)$. The §1.4 (SVD) section of the book writes the decomposition exactly this way: as a sum of rank-1 outer products $u_i v_i^\top$, each scaled by its singular value $\sigma_i$.

Truncating that sum to the first $k$ terms gives the **best possible rank-$k$ approximation** of $A$ in both the spectral and Frobenius norms — this is the Eckart–Young theorem:

$$
A_k = \sum_{i=1}^{k} \sigma_i \, u_i v_i^\top, \qquad \|A - A_k\|_F = \sqrt{\sum_{i=k+1}^{r} \sigma_i^2}.
$$

In other words, the reconstruction error after keeping $k$ singular values is completely determined by the *tail* of the singular value spectrum — nothing else. This is also where the condition number $\kappa(A) = \sigma_{\max}/\sigma_{\min}$ comes from: it measures how much a small input perturbation can be amplified by $A$.

In [ ]:
def economy_svd(A):
    """Return U, s, Vt with only r = rank(A) columns in U and rows in Vt."""
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    return U, s, Vt


def low_rank_approx(A, k):
    """Best rank-k approximation via truncated SVD (Eckart-Young theorem)."""
    U, s, Vt = economy_svd(A)
    return (U[:, :k] * s[:k]) @ Vt[:k, :]


def condition_number(A):
    """sigma_max / sigma_min -- measures numerical sensitivity to perturbations."""
    s = np.linalg.svd(A, compute_uv=False)
    return s[0] / s[-1]


# Build a synthetic matrix with a clearly decaying singular-value spectrum,
# similar to what you would see in a trained weight matrix's update.
m, n = 40, 30
true_rank3 = np.random.randn(m, 3) @ np.random.randn(3, n)
noise = 0.05 * np.random.randn(m, n)
A = true_rank3 + noise

U, s, Vt = economy_svd(A)
print("A shape:", A.shape)
print("Singular values (first 10):", s[:10])
print("Condition number kappa(A):", condition_number(A))

## What we are about to compute

`A` was built as a rank-3 signal (`true_rank3`) plus small dense noise, so we expect the first three singular values to dominate and the rest to be small but not exactly zero. We will:

1. Reconstruct `A` at ranks $k = 1, 2, 3$ using `low_rank_approx`.
2. Confirm the Eckart-Young identity numerically: the Frobenius reconstruction error at rank $k$ should match $\sqrt{\sum_{i>k} \sigma_i^2}$ to numerical precision.
3. Sweep $k$ from $1$ to $\min(m,n)$ and plot reconstruction error against the singular-value tail energy, so the two curves visibly coincide.

In [ ]:
def frobenius_error(A, A_k):
    return np.linalg.norm(A - A_k, ord="fro")


def tail_energy(s, k):
    """sqrt(sum of squares of singular values after the k-th) -- the Eckart-Young bound."""
    return np.sqrt(np.sum(s[k:] ** 2))


ranks_to_show = [1, 2, 3]
reconstructions = {k: low_rank_approx(A, k) for k in ranks_to_show}

print("Rank-k reconstruction error vs. Eckart-Young tail-energy bound")
print("-" * 62)
for k in ranks_to_show:
    err = frobenius_error(A, reconstructions[k])
    bound = tail_energy(s, k)
    print(f"k={k}:  ||A - A_k||_F = {err:.6f}   sqrt(sum sigma_i^2, i>k) = {bound:.6f}   "
          f"match = {np.isclose(err, bound)}")

max_k = min(m, n)
errors = np.array([frobenius_error(A, low_rank_approx(A, k)) for k in range(1, max_k + 1)])
bounds = np.array([tail_energy(s, k) for k in range(1, max_k + 1)])

## Visualizing the approximation

The left panel shows the original matrix `A` next to its rank-1, rank-2, and rank-3 reconstructions as heatmaps — by rank 3 the reconstruction should already look visually indistinguishable from `A`, since that is the true signal rank. The right panel plots reconstruction error against $k$ together with the Eckart-Young tail-energy bound; the two curves should overlap almost exactly, and the elbow at $k=3$ is the signature of a matrix whose "true" information content is much lower rank than its shape suggests.

In [ ]:
fig = plt.figure(figsize=(12, 4.5))
gs = GridSpec(1, 2, width_ratios=[1.4, 1], figure=fig)

# Left: heatmaps of A and its rank-k reconstructions
gs_left = gs[0].subgridspec(1, 4, wspace=0.15)
vmax = np.abs(A).max()
titles = ["A (original)"] + [f"rank-{k}" for k in ranks_to_show]
matrices = [A] + [reconstructions[k] for k in ranks_to_show]
for i, (title, M) in enumerate(zip(titles, matrices)):
    ax = fig.add_subplot(gs_left[i])
    ax.imshow(M, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

# Right: reconstruction error vs k, overlaid with the Eckart-Young bound
ax2 = fig.add_subplot(gs[1])
ks = np.arange(1, max_k + 1)
ax2.plot(ks, errors, "o-", label=r"$\|A - A_k\|_F$", color="C0", markersize=4)
ax2.plot(ks, bounds, "--", label=r"$\sqrt{\sum_{i>k}\sigma_i^2}$", color="C3", linewidth=2)
ax2.axvline(3, color="gray", linestyle=":", linewidth=1, label="true rank = 3")
ax2.set_xlabel("k (rank of approximation)")
ax2.set_ylabel("Frobenius error")
ax2.set_title("Reconstruction error vs. singular-value tail")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"Error at k=3: {errors[2]:.4f}  |  Error at k=max_k: {errors[-1]:.4e} (should be ~0)")
print("The elbow at k=3 confirms A's information content matches the rank-3 signal it was built from.")

## From toy demo to production ML: LoRA

This rank-$k$ truncation is not just a textbook exercise — it is, almost literally, what **LoRA (Low-Rank Adaptation)** does when fine-tuning large language models. Instead of updating a full weight matrix $W \in \mathbb{R}^{d \times k}$ during fine-tuning (which for a model like Llama-3's attention projections means millions of parameters per layer), LoRA freezes $W$ and learns a *low-rank update* $\Delta W = B A$ where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$ with $r \ll \min(d, k)$ — often $r \in \{4, 8, 16\}$ against hidden dimensions in the thousands.

The justification is the same Eckart-Young intuition we just verified numerically: empirically, the *useful* update to a pretrained weight matrix during task adaptation tends to have low "intrinsic rank" — most of its energy concentrates in a handful of singular directions, exactly like the tail of `s` decayed sharply after index 3 in our synthetic `A`. That is why practitioners can fine-tune a multi-billion-parameter model by training only a few million low-rank parameters and still recover most of the performance of full fine-tuning: they are betting that $\Delta W$ looks like our `low_rank_approx(A, k)` for small $k$, not like a full-rank, dense update.

In [ ]:
def grad_squared_norm(A, w, b):
    """
    f(w) = ||Aw - b||^2
    grad f(w) = 2 A^T (Aw - b)
    """
    return 2 * A.T @ (A @ w - b)


def numerical_jacobian(f, x, h=1e-5):
    """Finite-difference Jacobian of f: R^n -> R^m at x."""
    fx = f(x)
    m, n = fx.size, x.size
    J = np.zeros((m, n))
    for j in range(n):
        xp = x.copy(); xp[j] += h
        xm = x.copy(); xm[j] -= h
        J[:, j] = (f(xp) - f(xm)) / (2 * h)
    return J


# Gradient check: this is the exact sanity-check pattern used in every
# chapter's companion code -- the book promises every chapter prints a
# gradient check below 1e-5 before trusting any analytic gradient.
w0 = np.random.randn(n)
b_vec = np.random.randn(m)

def f_scalar(w):
    return np.array([np.sum((A @ w - b_vec) ** 2)])

analytic_grad = grad_squared_norm(A, w0, b_vec)
numeric_grad = numerical_jacobian(f_scalar, w0).flatten()

rel_error = np.linalg.norm(analytic_grad - numeric_grad) / np.linalg.norm(analytic_grad)
print("Analytic grad (first 5):", analytic_grad[:5])
print("Numeric  grad (first 5):", numeric_grad[:5])
print(f"Relative error: {rel_error:.2e}")
print(f"Gradient check PASSED (< 1e-5): {rel_error < 1e-5}")

## Exercise

1. The Eckart-Young theorem in the SVD section states that $A_k$ is the best rank-$k$ approximation of $A$ under *both* the spectral norm and the Frobenius norm. We only verified the Frobenius-norm identity above. Extend the code from the reconstruction-error cell to also compute $\|A - A_k\|_2$ (the spectral norm, i.e. the largest singular value of the residual) for each $k$, and confirm numerically that $\|A - A_k\|_2 = \sigma_{k+1}$. If your two curves diverge, work through the numerical-stability discussion in §1.7 before assuming a bug in your SVD call.

2. `grad_squared_norm` implements $\nabla f(w) = 2A^\top(Aw - b)$ from the gradients-of-quadratics section. Derive, on paper, the gradient of the *regularized* objective $f(w) = \|Aw - b\|^2 + \lambda \|w\|^2$, implement it, and reuse `numerical_jacobian` to gradient-check your implementation for a few values of $\lambda$ (including $\lambda = 0$, which should reduce exactly to `grad_squared_norm`). Run this finite-difference check before trusting any new gradient you write for the rest of the book.